# Pandas — Exercises

**Companion to deck 03 (Pandas Basics).** Run cells in order.

Each problem ships:
- a TODO cell — write your code
- an assertion cell — checks your answer
- a hint cell (run only if stuck)
- a solution cell at the bottom (don't peek too early)

Rule: **no Python `for` loop over rows**. One-liners only.

<a href="https://colab.research.google.com/github/Petkub/MachineLearningLab/blob/main/colab_exercises/03_pandas.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup — load real Titanic

We use the seaborn copy of Titanic — same shape as the Kaggle one, no auth needed.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns

df = sns.load_dataset('titanic')
# rename a couple of columns to match the Kaggle naming
df = df.rename(columns={'pclass': 'Pclass', 'sex': 'Sex', 'age': 'Age',
                        'sibsp': 'SibSp', 'parch': 'Parch', 'fare': 'Fare',
                        'survived': 'Survived', 'embarked': 'Embarked'})
print(df.shape)
df.head()

---
## Problem 01 — pick the right inspect call

A mentor walks up and asks **three quick questions**. Each maps to **one** pandas inspect tool you've already seen. Don't compute anything by hand — find the right shortcut.

Mentor's questions:
1. "How many rows and columns?" → save the **rows count** in `n_rows`.
2. "Any column missing values? Which one is worst?" → save the column **name** with the most NaN in `worst_col`.
3. "What's the spread of `Fare` — min, max, mean, std?" → save the **mean Fare** in `mean_fare`.

Pick the right one-liner per question. No loops.

In [ ]:
# TODO — fill in three values
n_rows    = ...
worst_col = ...
mean_fare = ...

print('rows:', n_rows)
print('worst NaN column:', worst_col)
print('mean Fare:', round(mean_fare, 2))

In [ ]:
assert n_rows == 891, f'expected 891 rows, got {n_rows}'
assert worst_col in df.columns, f'worst_col not a real column: {worst_col}'
assert df[worst_col].isnull().sum() == df.isnull().sum().max(), 'worst_col is not the column with most NaN'
assert abs(mean_fare - 32.20) < 1.0, f'mean Fare looks off: {mean_fare}'
print('Q1 ok')

<details><summary>Hint</summary>

Three different inspect tools, each ships ready-made:
- A row+col count: one **attribute** on the DataFrame returns the `(rows, cols)` tuple. No counting needed.
- Worst-NaN column: `df.isnull().sum()` gives a Series of counts per column. To get the **label of the maximum** (not the value), there's an `idxmax`-style method on Series.
- Spread of one column: `df.describe()` shows min/max/mean/std for numeric columns. Or pull `Fare` first and call its `.mean()`.

</details>

---
## Problem 02 — pull out rows that match

Find children (`Age < 12`) traveling in third class (`Pclass == 3`).

**Trap.** Python's `and` does *not* work on Series. Use `&` and parens around each clause.

Tasks:
1. Build boolean Series `mask` — True where both conditions hold.
2. Slice rows into `kids3`.
3. Keep only `Name` and `Age` columns from `kids3` → `kids3_short`.

*Note: seaborn version uses `who` instead of `Name`. We'll use the existing `who` column for this exercise.*

In [ ]:
# TODO
mask = ...
kids3 = ...
kids3_short = ...   # keep only ['who', 'Age']

print('count:', mask.sum())
kids3_short.head()

In [ ]:
assert mask.dtype == bool, 'mask must be a boolean Series'
assert mask.sum() > 0, 'mask matches zero rows — check logic'
assert (kids3['Age'] < 12).all(), 'kids3 has rows where Age >= 12'
assert (kids3['Pclass'] == 3).all(), 'kids3 has rows not in Pclass 3'
assert list(kids3_short.columns) == ['who', 'Age'], f'columns wrong: {list(kids3_short.columns)}'
print('Q2 ok')

<details><summary>Hint</summary>

Two conditions, both must hold. **Trap**: Python's word-form connectors fail on Series — pandas needs the bitwise version, and each clause must be wrapped in parens because of operator precedence. To slice a DataFrame by a boolean Series: pass that Series back into `df[...]`. To pick a subset of columns: pass a *list of names*, not a single string.

</details>

---
## Problem 03 — pick the right fill strategy per column

Three columns have NaN. Each needs a **different** repair strategy. You decide which.

| Column | Fix to apply |
|---|---|
| `Age` | numeric, roughly bell-shaped → fill with **mean** |
| `Fare` | numeric, heavy-tailed (outliers up to 500) → fill with **median** (robust) |
| `Embarked` | categorical with 3 values → fill with **most common** value |

Tasks:
1. Save count of NaN before any fill in `total_nan_before`.
2. Apply the three fills (assign back to the columns).
3. Save count of NaN after in `total_nan_after`. Should be 0.

**Think.** Mean for outlier-heavy data drags the fill toward extremes. Mode for categorical because mean of strings makes no sense. Same idea, different shape per column.

In [ ]:
# TODO
total_nan_before = ...

df['Age']      = ...   # fill with mean
df['Fare']     = ...   # fill with median
df['Embarked'] = ...   # fill with most common

total_nan_after = ...

print('NaN before:', total_nan_before)
print('NaN after: ', total_nan_after)

In [ ]:
assert total_nan_before > 0, 'expected some NaN before fill'
assert total_nan_after == 0, f'NaN still present: {total_nan_after}'
assert df['Age'].isnull().sum() == 0
assert df['Fare'].isnull().sum() == 0
assert df['Embarked'].isnull().sum() == 0
print('Q3 ok')

<details><summary>Hint</summary>

Total NaN across the whole frame: `df.isnull().sum().sum()` — first sum reduces per column, second sum collapses across columns.

For each fill, the same `.fillna(value)` pattern with a different `value`:
- mean of a Series → `.mean()` (skips NaN)
- median → `.median()` (skips NaN, ignores extreme outliers)
- most common string → `.mode()[0]` (mode returns a Series; take first)

Don't forget to **assign back** to `df['col']`.

</details>

---
## Problem 04 — did class matter for survival?

Compute survival rate per Pclass.

Tasks:
1. Save Series `rate` — survival rate per Pclass (mean of 0/1 column = rate).
2. Save the safest Pclass (highest rate) in `safest_class`.
3. Save the spread `gap` = max rate − min rate.

In [ ]:
# TODO
rate = ...
safest_class = ...
gap = ...

print(rate)
print('safest:', safest_class, '| gap:', round(gap, 3))

In [ ]:
assert isinstance(rate, pd.Series), 'rate must be a Series'
assert safest_class == 1, f'expected safest_class=1, got {safest_class}'
assert abs(gap - 0.39) < 0.05, f'gap looks off: {gap}'
print('Q4 ok')

<details><summary>Hint</summary>

"Per X" → group-by. Shape is: split by KEY, then take a stat of METRIC. Mean of a 0/1 column equals the rate — no manual division. To find which index has the maximum value of a Series, look for an `idxmax`-style method (returns the *label*, not the position). Spread = max minus min.

</details>

---
## Problem 05 — derived columns

Build a family-size column from `SibSp` + `Parch`. Then count solo travelers.

Tasks:
1. Add column `FamilySize = SibSp + Parch + 1` (passenger themselves counts).
2. Add boolean column `IsAlone = (FamilySize == 1)`.
3. Save `n_alone` = count of alone passengers, and `frac_alone` = fraction.

In [ ]:
# TODO
df['FamilySize'] = ...
df['IsAlone']    = ...
n_alone    = ...
frac_alone = ...

print('alone:', n_alone, '| fraction:', round(frac_alone, 3))
df[['SibSp', 'Parch', 'FamilySize', 'IsAlone']].head()

In [ ]:
assert (df['FamilySize'] >= 1).all(), 'FamilySize should be >= 1 (counts the passenger)'
assert df['IsAlone'].dtype == bool, 'IsAlone should be boolean'
assert n_alone == int(df['IsAlone'].sum()), 'n_alone count mismatch'
assert abs(frac_alone - 0.60) < 0.05, f'frac_alone looks off: {frac_alone}'
print('Q5 ok')

<details><summary>Hint</summary>

Vectorized arithmetic on Series — `+` works element-wise, no loop. A comparison like `series == 1` already returns a boolean Series; assign it directly. Counting True values: a boolean Series treats True as 1 — sum it. Fraction = count divided by total rows (`len(df)`).

</details>

---
## Problem 06 — filter, then split, then compare

Among adult women (`Age >= 18` AND `Sex == 'female'`), how big is the fare gap between 1st class and the average of 2nd+3rd?

Tasks:
1. Build `aw` — adult women only.
2. From `aw`, build `fare_by_class` — Series of mean Fare per Pclass.
3. Compute `fare_gap` = mean Fare for Pclass 1 — average of (mean Fare for Pclass 2, mean Fare for Pclass 3). Use the values from `fare_by_class`; you can read individual entries with `fare_by_class[1]`, `fare_by_class[2]`, etc.

In [ ]:
# TODO
aw = ...
fare_by_class = ...
fare_gap = ...

print(fare_by_class)
print('fare_gap:', round(fare_gap, 2))

In [ ]:
assert (aw['Sex'] == 'female').all(), 'aw contains non-female rows'
assert (aw['Age'] >= 18).all(), 'aw contains rows with Age < 18'
assert fare_gap > 50, f'fare_gap suspiciously small: {fare_gap}'
print('Q6 ok — adult women in 1st class paid roughly', round(fare_by_class.loc[1] / fare_by_class.loc[3], 1), 'times more than 3rd class')

<details><summary>Hint</summary>

Three stages, each shrinks the data:
1. Filter once (mask with `&` and parens) → adult women.
2. Group by Pclass, take mean of Fare → Series indexed by class number.
3. Read individual entries with `series[label]` (Series indexed by Pclass values 1, 2, 3). Average two values: `(a + b) / 2`.

Don't try to filter inside the groupby; keep stages separate.

</details>

---
## Problem 07 — find the bugs

The cell below tries to "count adult women in 1st class who survived." It runs but gives the wrong answer. **Find and fix three bugs.**

Things to look for: wrong logical operator, missing parentheses, target column accidentally used as a feature, wrong axis on a reduction.

In [ ]:
# BROKEN — fix three things, save the right count in `n_q7`.
#
# Goal: count rows that are adult (Age >= 18) AND female AND in 1st class AND survived.
#
# Three bugs to find:
#  - wrong logical connector
#  - missing parens (operator precedence)
#  - one of the comparisons has wrong value / wrong direction

mask = df['Sex'] == 'male' and df['Age'] >= 18 & df['Pclass'] == 1 & df['Survived'] == 1
n_q7 = mask.sum()
print('count:', n_q7)

In [ ]:
expected = ((df['Sex'] == 'female') & (df['Age'] >= 18) & (df['Pclass'] == 1) & (df['Survived'] == 1)).sum()
assert n_q7 == expected, f'still buggy — your count {n_q7}, expected {expected}'
print('Q7 ok — count =', n_q7)

<details><summary>Hint</summary>

Three independent failures:
- `and`/`or` work on Python booleans, not on Series. Pandas needs the bitwise versions.
- Even with the right operators, comparisons bind looser than `&` — every clause must be wrapped in its own `(...)`.
- One of the value comparisons doesn't match the goal stated in the prompt. Re-read: "adult, **female**, 1st class, survived."

</details>

---
## Solutions (don't peek too early)

<details><summary>Show all solutions</summary>

```python
# Q1
n_rows = df.shape[0]
worst_col = df.isnull().sum().idxmax()
mean_fare = df['Fare'].mean()

# Q2
mask = (df['Age'] < 12) & (df['Pclass'] == 3)
kids3 = df[mask]
kids3_short = kids3[['who', 'Age']]

# Q3
total_nan_before = df.isnull().sum().sum()
df['Age']      = df['Age'].fillna(df['Age'].mean())
df['Fare']     = df['Fare'].fillna(df['Fare'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
total_nan_after = df.isnull().sum().sum()

# Q4
rate = df.groupby('Pclass')['Survived'].mean()
safest_class = rate.idxmax()
gap = rate.max() - rate.min()

# Q5
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone']    = df['FamilySize'] == 1
n_alone    = int(df['IsAlone'].sum())
frac_alone = n_alone / len(df)

# Q6
aw = df[(df['Age'] >= 18) & (df['Sex'] == 'female')]
fare_by_class = aw.groupby('Pclass')['Fare'].mean()
fare_gap = fare_by_class[1] - (fare_by_class[2] + fare_by_class[3]) / 2

# Q7 — fixes:
#   - 'and' → '&'
#   - wrap each clause in parens
#   - 'male' → 'female'
mask = (df['Sex'] == 'female') & (df['Age'] >= 18) & (df['Pclass'] == 1) & (df['Survived'] == 1)
n_q7 = mask.sum()
```
</details>

## Recap

Pattern across all 7: **read what's there → filter / group → summarize**. Same shape every time. Memorize the verbs:

| verb | call |
|---|---|
| inspect | `.shape`, `.dtypes`, `.head()`, `.info()`, `.describe()` |
| filter | `df[mask]` with boolean Series |
| repair | `.fillna(...)`, `.dropna()` |
| summarize | `.groupby(KEY)[METRIC].agg(...)` |
| derive | `df['new'] = expr` |
